In [3]:
spark.stop()

In [1]:
import sys
sys.path.append('/home/azureuser/prathyusha/Kearney/prathyusha')
from utils import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark = instantiate_spark_sedona("60g")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/18 09:45:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/18 09:45:59 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
26/02/18 09:45:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/18 09:45:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark Session and SedonaContext have been successfully initiated.


In [2]:
loc_jan_df = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet('abfss://propheus-data-staging@propheusdatabay.dfs.core.windows.net/propheus-veraset-data/movement_v2/2026/*')

# loc_jan_df.count()

In [3]:
loc_jan_df.printSchema()

root
 |-- caid: string (nullable = true)
 |-- utc_timestamp: timestamp (nullable = true)
 |-- horizontal_accuracy: string (nullable = true)
 |-- id_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- iso_country_code: string (nullable = true)
 |-- quality_fields: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- geo_fields: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [4]:
loc_jan_df.show(5, truncate=False)

+----------------------------------------------------------------+-----------------------+-------------------+-------+---------------+---------+---------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------+
|caid                                                            |utc_timestamp          |horizontal_accuracy|id_type|ip_address     |latitude |longitude|iso_country_code|quality_fields                                                                                                                                                                                                                        |geo_fields                            

In [6]:
loc_jan_df = loc_jan_df.withColumn("latitude", col("latitude").cast(DoubleType())) \
                .withColumn("longitude", col("longitude").cast(DoubleType()))
loc_jan_df = loc_jan_df.withColumn("gh6", geohash_encode(col("latitude"), col("longitude"), lit(6))) \
                .withColumn("gh5", substr(col("gh6"), lit(1), lit(5))) \
                    .withColumn('quadkey', lat_lon_to_quadkey(col("latitude"), col("longitude"), lit(16)))



In [7]:
loc_jan_df.show()

+--------------------+--------------------+-------------------+-------+---------------+------------------+------------------+----------------+--------------------+--------------------+--------------------+------+-----+----------------+
|                caid|       utc_timestamp|horizontal_accuracy|id_type|     ip_address|          latitude|         longitude|iso_country_code|      quality_fields|          geo_fields|            Province|   gh6|  gh5|         quadkey|
+--------------------+--------------------+-------------------+-------+---------------+------------------+------------------+----------------+--------------------+--------------------+--------------------+------+-----+----------------+
|56870bef01ce5ec01...|2026-01-25 21:32:...|                0.0|   aaid|182.232.201.201|            6.7731|          101.3134|              TH|{ping_sink_matche...|{region -> pattan...|    pattani province|w30n93|w30n9|1322302200203211|
|b14b5a83e48c058dc...|2026-01-25 04:09:...|             

+----------------+------------+--------------------+------------------+-----------------------+------------------+--------------------+---------------+
|         quadkey|   adm1_name|Pop_nso_to_hdx_ratio|   hdx_Quadkey_Pop|nso_adjusted_population|sum_floor_area_sqm|Qk_pop_to_hh_sampled|Qk_estimated_hh|
+----------------+------------+--------------------+------------------+-----------------------+------------------+--------------------+---------------+
|1322121030100010|amnatcharoen|  1.3095653425533569| 142.0465920000001|     186.01929391101703| 76090.12173170949|   2.449373527632673|           76.0|
|1322121021232330|amnatcharoen|  1.3095653425533569|201.32532000000032|      263.6486616504646| 62566.12192884956|   2.488199875170112|          106.0|
|1322121030102232|amnatcharoen|  1.3095653425533569|135.38815800000015|     177.29963950893818| 54976.98828964075|   2.523829708720109|           70.0|
|1322121021312003|amnatcharoen|  1.3095653425533569|199.33478199999976|      261.0419220

In [6]:
quadkey_mapping = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/thailand_quadkeys_w_subdistrict")
quadkey_mapping.show()

+----------------+--------------------+--------------------+-----------------+------------+---------+--------------------+
|         quadkey|    quadkey_geometry|    quadkey_centroid|        adm3_name|   adm2_name|adm1_name|            geometry|
+----------------+--------------------+--------------------+-----------------+------------+---------+--------------------+
|1322122012022123|POLYGON ((101.980...|POINT (101.983337...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012022132|POLYGON ((101.986...|POINT (101.988830...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012022133|POLYGON ((101.991...|POINT (101.994323...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012023022|POLYGON ((101.997...|POINT (101.999816...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012023023|POLYGON ((102.002...|POINT (102.005310...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|132212201202302

In [8]:
final_df.count()

981682

In [7]:
join = final_df.select('quadkey', 'nso_adjusted_population', 'Qk_pop_to_hh_sampled', 'Qk_estimated_hh').join(
    quadkey_mapping.select('quadkey', 'adm2_name', 'adm1_name'),
    on = 'quadkey',
    how = 'inner'
)
join.count()

981682

In [9]:
join.show()

+----------------+-----------------------+--------------------+---------------+-------------------+------------+
|         quadkey|nso_adjusted_population|Qk_pop_to_hh_sampled|Qk_estimated_hh|          adm2_name|   adm1_name|
+----------------+-----------------------+--------------------+---------------+-------------------+------------+
|1322010330330112|       3.85783809004375|  3.0752889070291864|            1.0|         Pang Mapha|Mae Hong Son|
|1322010331201320|      2.149387830642142|   3.665291787562955|            1.0|         Pang Mapha|Mae Hong Son|
|1322010331201323|     3.1295666306991308|   2.997189388366925|            1.0|         Pang Mapha|Mae Hong Son|
|1322010331202200|       9.18006211671745|   2.760167183741254|            3.0|         Pang Mapha|Mae Hong Son|
|1322010331222012|      3.089849985643525|  3.3434753253878804|            1.0|         Pang Mapha|Mae Hong Son|
|1322010331232100|      5.355036234751846|   2.965004625512987|            2.0|         Pang Map

In [10]:
join_grouped = join.groupBy(col('adm1_name'), col('adm2_name')).agg(
    sum('nso_adjusted_population').alias('nso_adjusted_hdx_population'),
    sum('Qk_estimated_hh').alias('Qk_estimated_hh'),
)
join_grouped.show()

+-------------------+-----------------+---------------------------+---------------+
|          adm1_name|        adm2_name|nso_adjusted_hdx_population|Qk_estimated_hh|
+-------------------+-----------------+---------------------------+---------------+
|         Chiang Rai|          Pa Daet|         24578.121527529758|         9413.0|
|                Nan|        Santi Suk|         13083.532156029301|         5314.0|
|     Kamphaeng Phet|  Pang Sila Thong|           27757.4511022764|         9986.0|
|            Bangkok|       Khlong San|         109129.78173475343|        48481.0|
|      Maha Sarakham|     Yang Sisurat|          28337.10887818958|        11342.0|
|           Saraburi|         Nong Don|         14834.771568171085|         5322.0|
|   Nong Bua Lam Phu|         Na Klang|          90339.88305616792|        30943.0|
|  Nakhon Ratchasima|      Chum Phuang|          79499.40946119941|        27680.0|
|           Buri Ram|    Non Din Daeng|          19530.01458633106|         

In [12]:
join_grouped.toPandas().to_csv('../Data/adm1_adm2_level_pop_hh_estimated.csv', index=False)
